# `silver_clean.ipynb` - Capa Silver (esquema, calidad y particionado)

Transforma los lotes crudos de Bronze en una tabla columnar de tipos
definidos, aplicando el **contrato de calidad de datos estricto** exigido
por el enunciado (requisito funcional 2): ningun registro que no pase la
validacion de esquema (temperatura fuera de `[-50.0, 60.0] C`, humedad
fuera de `[0.0, 100.0] %`, precipitacion negativa, etc.) se escribe en la
tabla Silver.

Aplica el **patron de cuarentena** de las convenciones Nter: los registros
invalidos nunca se descartan silenciosamente, se segregan en
`data/silver/_cuarentena/cuarentena.parquet` junto con el motivo de
rechazo, y esa tabla **acumula entre ejecuciones** (no se trunca), igual
que se decidio en GeoStat para `tb_cuarentena_geodatos`.

> **Decision de diseno (idempotencia)**: cada particion de Silver
> (`ciudad`/`anio`/`mes`) se reescribe por completo en cada ejecucion a
> partir de la union de lo que ya habia mas los nuevos registros validos,
> deduplicando por `(ciudad, fecha_hora_utc)` y quedandonos con el
> registro de ingesta mas reciente. Esto garantiza que repetir el pipeline
> con los mismos datos de entrada no duplica filas ni altera las metricas
> analiticas (requisito de idempotencia del enunciado).

> **Decision de diseno (rango de presion)**: el enunciado no fija un rango
> explicito para `surface_pressure`. Se usa `[850.0, 1085.0] hPa`
> (`RANGO_PRESION_HPA` en `config.yaml`), que cubre con margen los
> extremos de presion a nivel de superficie registrados en la Peninsula
> Iberica.

> Depende de: `config.ipynb`, `logging_config.ipynb`, `bronze_ingest.ipynb`.

In [ ]:
import os
if os.getcwd().endswith("climate_engine"):
    os.chdir("..")
%run climate_engine/config.ipynb
%run climate_engine/logging_config.ipynb
%run climate_engine/bronze_ingest.ipynb

In [ ]:
import glob
import json
from pathlib import Path

import pandas as pd
import pandera.pandas as pa
from pandera.pandas import Column, Check, DataFrameSchema

## Funcion principal

In [ ]:
# --- Esquema de calidad de datos (Silver) ---
ESQUEMA_SILVER = DataFrameSchema(
    {
        "ciudad": Column(str, Check.isin(list(UBICACIONES.keys())), nullable=False),
        "fecha_hora_utc": Column("datetime64[ns]", nullable=False),
        "temperature_2m": Column(float, Check.in_range(*RANGO_TEMPERATURA_C), nullable=False),
        "relative_humidity_2m": Column(float, Check.in_range(*RANGO_HUMEDAD_PCT), nullable=False),
        "precipitation": Column(float, Check.in_range(*RANGO_PRECIPITACION_MM), nullable=False),
        "wind_speed_10m": Column(float, Check.in_range(*RANGO_VIENTO_KMH), nullable=False),
        "surface_pressure": Column(float, Check.in_range(*RANGO_PRESION_HPA), nullable=False),
        "id_lote": Column(str, nullable=False),
        "origen": Column(str, Check.isin(["API", "FALLBACK"]), nullable=False),
        "timestamp_ingesta_utc": Column(str, nullable=False),
    },
    coerce=True,
    strict=False,
)


def aplanar_bronze_ciudad(ruta_bronze):
    """Lee un fichero Bronze (sobre con metadata + datos_api) y lo aplana
    a un DataFrame en formato largo (una fila por hora), con una columna
    por variable meteorologica.

    Devuelve: DataFrame con columnas crudas (antes de tipar/validar).
    """
    with open(ruta_bronze, "r", encoding="utf-8") as f:
        registro = json.load(f)

    meta = registro["metadata"]
    horario = registro["datos_api"]["hourly"]
    df = pd.DataFrame({variable: horario[variable] for variable in API_VARIABLES_HORARIAS})
    df.insert(0, "fecha_hora_utc", horario["time"])
    df.insert(0, "ciudad", meta["ciudad"])
    df["id_lote"] = meta["id_lote"]
    df["origen"] = meta["origen"]
    df["timestamp_ingesta_utc"] = meta["timestamp_ingesta_utc"]
    return df


def aplanar_ingesta(resumen_bronze):
    """Aplana todos los ficheros Bronze escritos en una ingesta (ver
    bronze_ingest.ejecutar_ingesta_bronze) y los concatena en un unico
    DataFrame crudo.
    """
    marcos = [
        aplanar_bronze_ciudad(ruta)
        for ruta in resumen_bronze["rutas"].values()
        if ruta is not None
    ]
    if not marcos:
        return pd.DataFrame()
    return pd.concat(marcos, ignore_index=True)


def validar_y_separar(df_crudo):
    """Aplica el contrato de calidad de datos (ESQUEMA_SILVER) al
    DataFrame crudo. Los registros que no cumplen el esquema se separan
    en un DataFrame de cuarentena con el motivo de rechazo; nunca se
    descartan.

    Devuelve: tupla (df_validos, df_cuarentena).
    """
    df = df_crudo.copy()
    df["fecha_hora_utc"] = pd.to_datetime(df["fecha_hora_utc"], utc=False)
    for columna in ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m", "surface_pressure"]:
        df[columna] = pd.to_numeric(df[columna], errors="coerce")

    try:
        ESQUEMA_SILVER.validate(df, lazy=True)
        return df, df.iloc[0:0].assign(motivo_rechazo=pd.Series(dtype=str))
    except pa.errors.SchemaErrors as errores:
        casos_fallidos = errores.failure_cases
        indices_invalidos = sorted(set(casos_fallidos["index"].dropna().astype(int)))

        motivos_por_indice = (
            casos_fallidos.dropna(subset=["index"])
            .assign(index=lambda d: d["index"].astype(int))
            .groupby("index")
            .apply(lambda g: "; ".join(f"{r.column}: {r.check}" for r in g.itertuples()))
        )

        df_cuarentena = df.loc[indices_invalidos].copy()
        df_cuarentena["motivo_rechazo"] = df_cuarentena.index.map(motivos_por_indice)
        df_validos = df.drop(index=indices_invalidos)
        return df_validos.reset_index(drop=True), df_cuarentena.reset_index(drop=True)


def escribir_cuarentena(df_cuarentena):
    """Anexa los registros en cuarentena a data/silver/_cuarentena/cuarentena.parquet.
    Nunca se trunca: se acumula el historial completo de rechazos entre
    ejecuciones (decision de diseno heredada de GeoStat).
    """
    if df_cuarentena.empty:
        return 0
    ruta = RUTA_CUARENTENA / "cuarentena.parquet"
    if ruta.exists():
        previo = pd.read_parquet(ruta)
        combinado = pd.concat([previo, df_cuarentena], ignore_index=True)
    else:
        combinado = df_cuarentena
    combinado.to_parquet(ruta, index=False)
    return len(df_cuarentena)


def escribir_silver(df_validos, forzar_particiones=False):
    """Escribe los registros validos en la capa Silver, particionados por
    ciudad/anio/mes, deduplicando por (ciudad, fecha_hora_utc) de forma
    idempotente: cada particion se reescribe por completo combinando lo
    ya existente con lo nuevo, quedandose con el registro mas reciente.

    Si forzar_particiones=True (opcion --forzar-particiones del CLI), se
    ignora el contenido previo de cada particion afectada y se reescribe
    solo con los datos nuevos: util para reconstruir una particion desde
    cero tras una correccion.

    Devuelve: numero de filas finales escritas (tras deduplicar).
    """
    if df_validos.empty:
        return 0

    df = df_validos.copy()
    df["anio"] = df["fecha_hora_utc"].dt.year
    df["mes"] = df["fecha_hora_utc"].dt.month

    total_filas_finales = 0
    for (ciudad, anio, mes), grupo_nuevo in df.groupby(["ciudad", "anio", "mes"]):
        directorio = RUTA_SILVER / f"ciudad={ciudad}" / f"anio={anio}" / f"mes={mes:02d}"
        directorio.mkdir(parents=True, exist_ok=True)
        ruta_particion = directorio / "datos.parquet"

        if ruta_particion.exists() and not forzar_particiones:
            previo = pd.read_parquet(ruta_particion)
            combinado = pd.concat([previo, grupo_nuevo.drop(columns=["anio", "mes"])], ignore_index=True)
        else:
            if ruta_particion.exists() and forzar_particiones:
                logger.info(f"Forzado de particion (--forzar-particiones): se reescribe {ruta_particion}")
            combinado = grupo_nuevo.drop(columns=["anio", "mes"])

        combinado = combinado.sort_values("timestamp_ingesta_utc").drop_duplicates(
            subset=["ciudad", "fecha_hora_utc"], keep="last"
        )
        combinado.to_parquet(ruta_particion, index=False)
        total_filas_finales += len(combinado)

    return total_filas_finales


def procesar_silver(resumen_bronze, forzar_particiones=False):
    """Orquesta la capa Silver completa: aplanar -> validar/separar ->
    escribir validos (particionado, idempotente salvo forzar_particiones)
    -> escribir cuarentena (acumulativo). Registra en el log el resumen
    de cifras, siguiendo el formato de las convenciones Nter.

    Devuelve: diccionario con las cifras de la ejecucion.
    """
    df_crudo = aplanar_ingesta(resumen_bronze)
    total_leido = len(df_crudo)
    if total_leido == 0:
        logger.warning("Silver: no hay datos Bronze que procesar en esta ejecucion.")
        return {"total_leido": 0, "validos": 0, "cuarentena": 0}

    df_validos, df_cuarentena = validar_y_separar(df_crudo)
    n_validos_final = escribir_silver(df_validos, forzar_particiones=forzar_particiones)
    n_cuarentena = escribir_cuarentena(df_cuarentena)

    logger.info(
        f"Silver finalizado. Total leido: {total_leido} | "
        f"Validos: {len(df_validos)} | Cuarentena: {n_cuarentena}"
    )
    return {
        "total_leido": total_leido,
        "validos": len(df_validos),
        "cuarentena": n_cuarentena,
        "filas_finales_en_silver": n_validos_final,
    }

## Prueba rapida

Se procesa el ultimo lote Bronze real disponible y, ademas, se inyecta a proposito un registro corrupto (temperatura de 999 C) para comprobar en vivo que el contrato de calidad lo envia a cuarentena sin descartarlo y sin contaminar la tabla Silver.

In [ ]:
df_crudo_prueba = aplanar_ingesta(resumen_bronze)
fila_corrupta = df_crudo_prueba.iloc[[0]].copy()
fila_corrupta["temperature_2m"] = 999.0
fila_corrupta["ciudad"] = df_crudo_prueba.iloc[0]["ciudad"]
df_crudo_prueba = pd.concat([df_crudo_prueba, fila_corrupta], ignore_index=True)

df_validos_prueba, df_cuarentena_prueba = validar_y_separar(df_crudo_prueba)

print(f"Filas crudas de entrada (incluye 1 corrupta inyectada a proposito): {len(df_crudo_prueba)}")
print(f"Filas validas:     {len(df_validos_prueba)}")
print(f"Filas en cuarentena: {len(df_cuarentena_prueba)}")
if not df_cuarentena_prueba.empty:
    print("\nDetalle de la cuarentena:")
    print(df_cuarentena_prueba[["ciudad", "fecha_hora_utc", "temperature_2m", "motivo_rechazo"]].to_string(index=False))

assert len(df_cuarentena_prueba) >= 1, "El registro corrupto inyectado deberia caer en cuarentena."
assert 999.0 not in df_validos_prueba["temperature_2m"].values, "El registro corrupto no debe llegar a los datos validos."
print("\nValidacion del contrato de calidad de datos: OK")

print("\n--- Ejecucion real completa de la capa Silver (sin el registro inyectado) ---")
resumen_silver = procesar_silver(resumen_bronze)
print(resumen_silver)

---
✅ **`silver_clean.ipynb` verificado/a.**